# Week 2.0 Notebook: HAIC, uv, And SLURM Basics

This notebook proves the laptop-safe part of the HAIC or Sherlock workflow. It records portable paths, prints cluster discovery commands, validates that SLURM directives stay at the top of a batch script, and finishes with the synthetic Week 2 data gate. It does not open SSH or run cluster jobs for you; it shows the exact commands to run later from the correct machine and directory.


## SSH And Cluster Command Map

This notebook is safe to run on a laptop because it prints cluster commands instead of opening an SSH connection. When you use HAIC or Sherlock for real, read command prompts as location labels. Do not type `laptop$`, `haic-login$`, or `haic-compute$`.

Before any real command, ask: which machine am I controlling, and which directory am I in? Use `hostname` and `pwd` whenever you are unsure.

| Example | Where you should be | Meaning |
| --- | --- | --- |
| `laptop$ ssh <sunetid>@<haic-login-host>` | Any directory on your laptop | Open a remote terminal on the HAIC or Sherlock login node. Use `sherlock.stanford.edu` when your instructions say to use Sherlock. |
| `haic-login$ hostname` | Any directory on the login node | Print the name of the machine you are currently using. This confirms you are no longer on your laptop. |
| `haic-login$ salloc ...` | Any directory on the login node | Ask SLURM for an interactive compute-node shell. After allocation, move to `$CODY_JEPA_ROOT` before repo commands. |
| `haic-login$ sbatch script.sbatch` | Usually `$CODY_JEPA_ROOT` on the login node | Submit a batch script. The work runs later on a compute node. |
| `haic-login$ squeue -u "$USER"` | Show your pending and running jobs. |
| `haic-login$ sacct -j <job_id> ...` | Inspect a completed job. Replace `<job_id>` with the numeric ID printed by `sbatch`. |
| `haic-compute$ exit` | Leave the compute allocation. If you are in an SSH login shell, `exit` again returns to your laptop. |

SSH gets you onto the login node. SLURM gets compute resources. Those are different steps, and most heavy commands in this project should run only after SLURM gives you compute resources. Stanford's resources overview lists public Sherlock partitions such as `normal`, `gpu`, `dev`, `bigmem`, and `owners`, and it also mentions `serc` for eligible SDSS users. Discover your own account access before using any partition name. Before saving site-specific batch scripts, run `mkdir -p scripts logs` from `$CODY_JEPA_ROOT` on the login node.


## Path Setup

This cell group proves that all generated files land under ignored `data/gaitlu-1m/` paths and that the notebook can find the repo root from any working directory. On HAIC or Sherlock, repo commands should still be run from `$CODY_JEPA_ROOT` unless a tutorial says otherwise. Large real data should live under `$SCRATCH` or approved project storage, not under `$HOME`.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import pickle
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pyproject.toml")


REPO_ROOT = find_repo_root()
GAITLU_ROOT = Path(os.environ.get("GAITLU_ROOT", REPO_ROOT / "data" / "gaitlu-1m"))
ARCHIVE_DIR = Path(os.environ.get("GAITLU_ARCHIVE_DIR", GAITLU_ROOT / "archives"))
RAW_DIR = Path(os.environ.get("GAITLU_EXTRACTED_DIR", GAITLU_ROOT / "raw"))
NOTEBOOK_MODE = os.environ.get("GAITLU_NOTEBOOK_MODE", "synthetic").lower()
if NOTEBOOK_MODE not in {"synthetic", "real"}:
    raise ValueError("GAITLU_NOTEBOOK_MODE must be 'synthetic' or 'real'")
SYNTHETIC_RAW_DIR = Path(os.environ.get("GAITLU_SYNTHETIC_RAW_DIR", GAITLU_ROOT / "synthetic_raw"))
MANIFEST_DIR = Path(os.environ.get("GAITLU_MANIFEST_DIR", GAITLU_ROOT / "manifests"))
DIAGNOSTICS_DIR = Path(os.environ.get("GAITLU_DIAGNOSTICS_DIR", GAITLU_ROOT / "diagnostics"))
PROBE_EXPORT_DIR = Path(os.environ.get("GAITLU_PROBE_EXPORT_DIR", GAITLU_ROOT / "probe_exports"))

for path in (ARCHIVE_DIR, MANIFEST_DIR, DIAGNOSTICS_DIR, PROBE_EXPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
if NOTEBOOK_MODE == "synthetic":
    SYNTHETIC_RAW_DIR.mkdir(parents=True, exist_ok=True)
elif not RAW_DIR.exists():
    raise FileNotFoundError(f"GAITLU_EXTRACTED_DIR does not exist: {RAW_DIR}")

print("repo", REPO_ROOT)
print("gaitlu_root", GAITLU_ROOT)
print("notebook_mode", NOTEBOOK_MODE)
print("raw_dir", RAW_DIR)
print("synthetic_raw_dir", SYNTHETIC_RAW_DIR)


## HAIC Or Sherlock Discovery Commands

These commands are meant for a HAIC or Sherlock login node. The notebook prints them instead of running them locally because this synthetic validation should also work on a laptop.


In [ ]:
discovery_commands = [
    "hostname",
    "whoami",
    "groups",
    "sacctmgr show assoc user=$USER format=Cluster,Account,Partition,QOS%30",
    "sinfo",
    "sinfo -o '%20P %8a %10l %10D %30G'",
    "sinfo --Node --long --partition=serc",
    "scontrol show partition",
    "df -h",
    "quota -s",
    "echo $SCRATCH",
    "echo $OAK",
    "module spider cuda 2>/dev/null || module avail cuda 2>&1",
]
print("Run these on HAIC or Sherlock, not inside this laptop smoke notebook:")
for command in discovery_commands:
    print("  ", command)


## Portable Variables

This cell records the environment-variable contract used by the tutorials. On HAIC or Sherlock, set these after discovering your actual storage location. Prefer `$SCRATCH/cody-jepa-data` for large temporary GaitLU data. Fall back to `$HOME/cody-jepa-data` only for small smoke tests.


In [ ]:
portable_env = {
    "CODY_JEPA_ROOT": "$HOME/cody-jepa",
    "CODY_JEPA_DATA": "$SCRATCH/cody-jepa-data if SCRATCH exists, otherwise $HOME/cody-jepa-data for smoke tests",
    "GAITLU_ARCHIVE_DIR": "$CODY_JEPA_DATA/gaitlu-1m/archives",
    "GAITLU_EXTRACTED_DIR": "$CODY_JEPA_DATA/gaitlu-1m/raw",
    "GAITLU_MANIFEST_DIR": "$CODY_JEPA_DATA/gaitlu-1m/manifests",
    "GAITLU_DIAGNOSTICS_DIR": "$CODY_JEPA_DATA/gaitlu-1m/diagnostics",
    "GAITLU_PROBE_EXPORT_DIR": "$CODY_JEPA_DATA/gaitlu-1m/probe_exports",
}
print(json.dumps(portable_env, indent=2))


## SLURM Script Shape

This cell proves that the example `#SBATCH` directives appear before any executable shell command.


In [ ]:
smoke_sbatch = """#!/bin/bash
#SBATCH --job-name=cody_haic_smoke
#SBATCH --time=00:10:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=2
#SBATCH --mem=8G
#SBATCH --output=logs/%x-%j.out
#SBATCH --error=logs/%x-%j.err

set -euo pipefail

cd "${CODY_JEPA_ROOT:?Set CODY_JEPA_ROOT before sbatch}"
uv run python - <<'PY'
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
PY
"""
script_lines = smoke_sbatch.splitlines()
first_executable = next(i for i, line in enumerate(script_lines) if line and not line.startswith("#"))
late_directives = [line for line in script_lines[first_executable:] if line.startswith("#SBATCH")]
assert not late_directives
script_path = DIAGNOSTICS_DIR / "haic_smoke.sbatch"
script_path.write_text(smoke_sbatch)
print(script_path)


## Synthetic Fixture

This cell group proves that the rest of Week 2 can be tested without the real encrypted GaitLU archive.


In [ ]:
def stable_int(text, seed=0):
    payload = f"{seed}:{text}".encode("utf-8")
    return int(hashlib.sha1(payload).hexdigest()[:16], 16)


def create_synthetic_gaitlu_fixture(raw_dir=None, variant="anonymized_sil"):
    if NOTEBOOK_MODE != "synthetic":
        raise RuntimeError("Refusing to create synthetic fixtures when GAITLU_NOTEBOOK_MODE is not 'synthetic'")
    raw_dir = SYNTHETIC_RAW_DIR if raw_dir is None else Path(raw_dir)
    root = raw_dir / variant
    root.mkdir(parents=True, exist_ok=True)
    for top in range(4):
        for mid in range(3):
            leaf = root / f"{top:03d}" / f"{mid:03d}" / "000"
            leaf.mkdir(parents=True, exist_ok=True)
            path = leaf / f"{top:03d}_{mid:03d}_000.pkl"
            if path.exists():
                continue
            frames = 18 + top + mid
            height, width = 64, 44
            arr = np.zeros((frames, height, width), dtype=np.uint8)
            for t in range(frames):
                x = (2 * t + 3 * top + mid) % (width - 8)
                y = 12 + 3 * mid
                arr[t, y:y + 24, x:x + 6] = 255
                arr[t, max(0, y - 5):y, min(width - 1, x + 2):min(width, x + 9)] = 160
            with path.open("wb") as f:
                pickle.dump(arr, f)
    return root


def discover_pkl_root(base):
    base = Path(base)
    for name in ("silhouette_cut_pkl", "anonymized_sil"):
        candidate = base / name
        if candidate.exists() and any(candidate.rglob("*.pkl")):
            return candidate
    pkls = sorted(base.rglob("*.pkl"))
    if not pkls:
        raise FileNotFoundError(f"No .pkl files under {base}")
    return Path(os.path.commonpath([str(p.parent) for p in pkls]))


def coerce_sequence(payload):
    if isinstance(payload, dict):
        for key in ("silhouette", "silhouettes", "frames", "imgs", "data", "seq"):
            if key in payload:
                return coerce_sequence(payload[key])
        for value in payload.values():
            try:
                return coerce_sequence(value)
            except Exception:
                continue
        raise ValueError("Dictionary payload did not contain an array-like sequence")
    arr = np.asarray(payload)
    arr = np.squeeze(arr)
    if arr.ndim == 2:
        arr = arr[None, :, :]
    if arr.ndim == 4 and arr.shape[-1] in (1, 3, 4):
        arr = arr[..., 0]
    if arr.ndim == 4 and arr.shape[1] in (1, 3, 4):
        arr = arr[:, 0, :, :]
    if arr.ndim != 3:
        raise ValueError(f"Expected sequence shape [T,H,W], got {arr.shape}")
    return arr


def read_pickle_sequence(path):
    with Path(path).open("rb") as f:
        payload = pickle.load(f)
    return coerce_sequence(payload)


def build_index(pkl_root):
    pkl_root = Path(pkl_root)
    rows = []
    for pkl_path in sorted(pkl_root.rglob("*.pkl")):
        rel = pkl_path.relative_to(pkl_root)
        shard_tokens = rel.parts[:-1]
        row = {
            "sequence_id": hashlib.sha1(str(rel).encode("utf-8")).hexdigest()[:16],
            "relative_pkl_path": str(rel),
            "shard_0": shard_tokens[0] if len(shard_tokens) > 0 else "",
            "shard_1": shard_tokens[1] if len(shard_tokens) > 1 else "",
            "shard_2": shard_tokens[2] if len(shard_tokens) > 2 else "",
            "shard_path": str(Path(*shard_tokens)) if shard_tokens else "",
        }
        try:
            arr = read_pickle_sequence(pkl_path)
            row.update({
                "num_frames": int(arr.shape[0]),
                "height": int(arr.shape[1]),
                "width": int(arr.shape[2]),
                "dtype": str(arr.dtype),
                "read_ok": True,
                "error": "",
            })
        except Exception as exc:
            row.update({
                "num_frames": 0,
                "height": 0,
                "width": 0,
                "dtype": "",
                "read_ok": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
        rows.append(row)
    return pd.DataFrame(rows)


def choose_proxy_split_key(index):
    for column in ("shard_0", "shard_1", "shard_2", "shard_path", "sequence_id"):
        values = [v for v in sorted(index[column].astype(str).unique()) if v]
        if 1 < len(values) < len(index):
            return column
    return "sequence_id"


def make_splits(index, seed=7, val_fraction=0.25):
    df = index[index["read_ok"]].copy().reset_index(drop=True)
    proxy_column = choose_proxy_split_key(df)
    df["proxy_split_key"] = df[proxy_column].astype(str)
    groups = sorted(df["proxy_split_key"].unique())
    if len(groups) == 1:
        val_groups = set()
    else:
        ranked = sorted(groups, key=lambda group: stable_int(group, seed=seed))
        n_val = max(1, min(len(groups) - 1, round(len(groups) * val_fraction)))
        val_groups = set(ranked[:n_val])
    df["split"] = np.where(df["proxy_split_key"].isin(val_groups), "val", "train")
    df["split_seed"] = seed
    df["split_proxy_column"] = proxy_column
    columns = [
        "sequence_id", "relative_pkl_path", "shard_0", "shard_1", "shard_2", "shard_path",
        "num_frames", "height", "width", "dtype", "read_ok", "error",
        "proxy_split_key", "split", "split_seed", "split_proxy_column",
    ]
    return df[columns].sort_values("sequence_id").reset_index(drop=True)


class GaitLUPklDataset(Dataset):
    def __init__(self, pkl_root, split_manifest, split, clip_length=16, seed=7):
        self.pkl_root = Path(pkl_root)
        self.rows = split_manifest[split_manifest["split"] == split].reset_index(drop=True)
        self.split = split
        self.clip_length = int(clip_length)
        self.seed = int(seed)
        self.epoch = 0
        if self.rows.empty:
            raise ValueError(f"No rows for split={split}")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.rows)

    def _start(self, sequence_id, num_frames):
        max_start = max(0, int(num_frames) - self.clip_length)
        if max_start == 0:
            return 0
        if self.split == "train":
            rng = random.Random(f"{self.seed}:{self.epoch}:{sequence_id}")
            return rng.randrange(max_start + 1)
        return stable_int(f"val:{sequence_id}:{self.clip_length}", seed=self.seed) % (max_start + 1)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        arr = read_pickle_sequence(self.pkl_root / row["relative_pkl_path"])
        start = self._start(row["sequence_id"], arr.shape[0])
        clip = arr[start:start + self.clip_length]
        if clip.shape[0] < self.clip_length:
            pad_count = self.clip_length - clip.shape[0]
            pad = np.repeat(clip[-1:, :, :], pad_count, axis=0)
            clip = np.concatenate([clip, pad], axis=0)
        clip = clip.astype(np.float32)
        if clip.max() > 1.0:
            clip = clip / 255.0
        video = torch.from_numpy(clip).unsqueeze(1).contiguous()
        return {
            "video": video,
            "sequence_id": row["sequence_id"],
            "split": row["split"],
            "proxy_label": row["proxy_split_key"],
        }


def run_week2_gate(pkl_root, seed=7, clip_length=16, batch_size=2):
    index = build_index(pkl_root)
    split_a = make_splits(index, seed=seed)
    split_b = make_splits(index, seed=seed)
    splits_reproduce = split_a.to_csv(index=False) == split_b.to_csv(index=False)

    val_a = GaitLUPklDataset(pkl_root, split_a, "val", clip_length=clip_length, seed=seed)
    val_b = GaitLUPklDataset(pkl_root, split_b, "val", clip_length=clip_length, seed=seed)
    batch_a = next(iter(DataLoader(val_a, batch_size=batch_size, shuffle=False, num_workers=0)))
    batch_b = next(iter(DataLoader(val_b, batch_size=batch_size, shuffle=False, num_workers=0)))

    validation_reproduces = torch.equal(batch_a["video"], batch_b["video"])
    video_shape = list(batch_a["video"].shape)
    shape_ok = len(video_shape) == 5 and video_shape[1:3] == [clip_length, 1]
    result = {
        "splits_reproduce": bool(splits_reproduce),
        "validation_batches_reproduce": bool(validation_reproduces),
        "batch_shape": video_shape,
        "shape_ok": bool(shape_ok),
    }
    assert result["splits_reproduce"], result
    assert result["validation_batches_reproduce"], result
    assert result["shape_ok"], result
    return result, split_a, batch_a


## Week 2 Gate

The gate passes only when splits reproduce exactly, validation batches reproduce exactly, and the loader returns `[B, T, C, H, W]`.


In [ ]:
PKL_ROOT = create_synthetic_gaitlu_fixture()
result, split_manifest, batch = run_week2_gate(PKL_ROOT)
print(json.dumps(result, indent=2))
assert batch["video"].ndim == 5
